Bước 2 — EDA (Khám phá dữ liệu)

In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv('../data/processed/movies_clean.csv')
ratings = pd.read_csv('../data/processed/ratings_clean.csv')

print("movies:", movies.shape)
print("ratings:", ratings.shape)
movies.head(3)

movies: (45429, 21)
ratings: (100004, 4)


,id,title,overview,genres,keywords,cast,crew,popularity,vote_average,vote_count,...,poster_path,genres_list,keywords_list,cast_list,director,release_year,genres_clean,keywords_clean,cast_clean,director_clean
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",21.946943,7.7,5415.0,...,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"['Animation', 'Comedy', 'Family']","['jealousy', 'toy', 'boy', 'friendship', 'frie...","['Tom Hanks', 'Tim Allen', 'Don Rickles']",John Lasseter,1995.0,"['animation', 'comedy', 'family']","['jealousy', 'toy', 'boy', 'friendship', 'frie...","['tomhanks', 'timallen', 'donrickles']",johnlasseter
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",17.015539,6.9,2413.0,...,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"['Adventure', 'Fantasy', 'Family']","['board game', 'disappearance', ""based on chil...","['Robin Williams', 'Jonathan Hyde', 'Kirsten D...",Joe Johnston,1995.0,"['adventure', 'fantasy', 'family']","['boardgame', 'disappearance', ""basedonchildre...","['robinwilliams', 'jonathanhyde', 'kirstendunst']",joejohnston
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",11.712900,6.5,92.0,...,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"['Romance', 'Comedy']","['fishing', 'best friend', 'duringcreditssting...","['Walter Matthau', 'Jack Lemmon', 'Ann-Margret']",Howard Deutch,1995.0,"['romance', 'comedy']","['fishing', 'bestfriend', 'duringcreditsstinge...","['waltermatthau', 'jacklemmon', 'ann-margret']",howarddeutch


In [3]:
# Phân bố rating
ratings['rating'].describe()

count    100004.000000
mean          3.543608
std           1.058064
min           0.500000
25%           3.000000
50%           4.000000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [4]:
import ast

# genres_list đang được lưu dạng chuỗi "['Animation', 'Comedy', ...]" khi đọc lại từ CSV
# cần parse lại bằng ast.literal_eval
movies['genres_list'] = movies['genres_list'].apply(ast.literal_eval)

# Đếm số phim theo từng thể loại
from collections import Counter
genre_counts = Counter([g for sublist in movies['genres_list'] for g in sublist])
genre_df = pd.DataFrame(genre_counts.items(), columns=['genre', 'count']).sort_values('count', ascending=False)
genre_df

,genre,count
6,Drama,20243
1,Comedy,13176
9,Thriller,7618
5,Romance,6730
7,Action,6590
10,Horror,4670
8,Crime,4304
17,Documentary,3930
3,Adventure,3490
12,Science Fiction,3042


In [5]:
# Phân bố năm phát hành
movies['release_year'].describe()

count    45345.000000
mean      1991.882280
std         24.053016
min       1874.000000
25%       1978.000000
50%       2001.000000
75%       2010.000000
max       2020.000000
Name: release_year, dtype: float64

In [6]:
# Số rating theo từng user (để đánh giá độ thưa)
ratings_per_user = ratings.groupby('userId').size()
print(ratings_per_user.describe())

# Số rating theo từng phim
ratings_per_movie = ratings.groupby('movieId').size()
print(ratings_per_movie.describe())

count     671.000000
mean      149.037258
std       231.226948
min        20.000000
25%        37.000000
50%        71.000000
75%       161.000000
max      2391.000000
dtype: float64
count    9066.000000
mean       11.030664
std        24.050800
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       341.000000
dtype: float64


In [7]:
# Độ thưa (sparsity) của ma trận user-item
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)

sparsity = 1 - (n_ratings / (n_users * n_movies))
print(f"Số user: {n_users}, Số phim có rating: {n_movies}, Số rating: {n_ratings}")
print(f"Độ thưa (sparsity): {sparsity:.4%}")

Số user: 671, Số phim có rating: 9066, Số rating: 100004
Độ thưa (sparsity): 98.3561%


Bước 2.2: Điều tra outlier + kiểm tra mapping + xác định ngưỡng lọc

In [8]:
# Điều tra phim có release_year bất thường (quá cũ, quá xa thực tế)
movies[movies['release_year'] < 1900][['id', 'title', 'release_date', 'release_year']]

,id,title,release_date,release_year
16245,49296,The Four Troublesome Heads,1898-01-01,1898.0
16250,195311,The Human Pyramid,1899-05-20,1899.0
17561,105158,Edison Kinetoscopic Record of a Sneeze,1894-01-09,1894.0
18906,159900,"Ella Lola, a la Trilby",1898-01-01,1898.0
18936,159907,"Turkish Dance, Ella Lola",1898-09-30,1898.0
...,...,...,...,...
45223,104700,The Astronomer's Dream,1898-01-01,1898.0
45226,104702,The Temptation of St. Anthony,1898-01-01,1898.0
45227,49295,An Up-to-Date Conjurer,1899-01-01,1899.0
45297,104704,The Devil in a Convent,1899-01-01,1899.0


In [9]:
# Kiểm tra mức độ khớp giữa ratings (movieId) và movies_clean (id/tmdbId) qua links_small
links_small = pd.read_csv('../data/raw/links_small.csv')

# Map movieId -> tmdbId
ratings_mapped = ratings.merge(links_small[['movieId', 'tmdbId']], on='movieId', how='left')
print("Rating không map được sang tmdbId:", ratings_mapped['tmdbId'].isnull().sum())

# Kiểm tra bao nhiêu tmdbId trong ratings thực sự tồn tại trong movies_clean
valid_tmdb_ids = set(movies['id'])
ratings_mapped['tmdbId'] = ratings_mapped['tmdbId'].fillna(-1).astype('int64')
matched = ratings_mapped['tmdbId'].isin(valid_tmdb_ids).sum()
print(f"Rating khớp được với movies_clean: {matched} / {len(ratings_mapped)} ({matched/len(ratings_mapped):.2%})")

Rating không map được sang tmdbId: 71
Rating khớp được với movies_clean: 99810 / 100004 (99.81%)


In [10]:
# Thử các ngưỡng lọc khác nhau cho số rating/phim, xem còn lại bao nhiêu phim
for min_ratings in [1, 3, 5, 10, 20]:
    n_movies_left = (ratings_per_movie >= min_ratings).sum()
    print(f"Ngưỡng >= {min_ratings} rating: còn {n_movies_left} phim ({n_movies_left/len(ratings_per_movie):.1%})")

Ngưỡng >= 1 rating: còn 9066 phim (100.0%)
Ngưỡng >= 3 rating: còn 4801 phim (53.0%)
Ngưỡng >= 5 rating: còn 3496 phim (38.6%)
Ngưỡng >= 10 rating: còn 2245 phim (24.8%)
Ngưỡng >= 20 rating: còn 1303 phim (14.4%)


In [11]:
# Xem phân bố popularity/vote_average theo dạng phân vị chi tiết hơn (để phát hiện outlier)
movies[['popularity', 'vote_average', 'vote_count']].quantile([0.01, 0.25, 0.5, 0.75, 0.95, 0.99])

,popularity,vote_average,vote_count
0.01,0.000766,0.0,0.00
0.25,0.385938,5.0,3.00
0.50,1.127377,6.0,10.00
0.75,3.678189,6.8,34.00
0.95,11.061757,7.8,434.00
0.99,17.009464,8.7,2184.44
